# 🌦️ SkyGuard AI — 1D-CNN Autoencoder Training in Google Colab

This notebook trains our lightweight 1D-CNN Autoencoder for edge-to-cloud anomaly detection on India's Automatic Weather Stations (Agra AWS).

### Pipeline Highlights:
1. **Load Baseline Diurnal Weather:** Ingests `train_baseline_normal.csv` (10,000 steps).
2. **Sliding Windows:** Formats data into 12-timestep windows (2 hours of history, 8 features).
3. **Train 1D-CNN Autoencoder:** Learns physical relationships between Temperature, Humidity, and Pressure.
4. **Threshold Calibration:** Calculates nominal MSE ($< 0.02$) vs Drift Anomaly ($> 0.042$).
5. **Export Deployable Artifacts:** Exports `autoencoder.onnx` (< 500 KB) and `scaler.json` for real-time inference.

### Step 1: Install Dependencies and Import Libraries

In [ ]:
!pip install -q torch onnx onnxruntime pandas numpy matplotlib seaborn

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

### Step 2: Load and Prepare Training & Test Datasets
*Note: Upload `train_baseline_normal.csv` and `test_fault_injections.csv` into the Colab files pane on the left.*

In [ ]:
train_file = "train_baseline_normal.csv"
test_file = "test_fault_injections.csv"

if not os.path.exists(train_file):
    raise FileNotFoundError(f"{train_file} not found. Please drag and drop it into the Colab file pane!")

df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file) if os.path.exists(test_file) else None

print(f"Train samples: {len(df_train)}")
display(df_train.head())

### Step 3: Feature Engineering & Scaling
We compute the temporal rates of change ($dT/dt$, $dP/dt$, $dRH/dt$) and normalize features using standard scaling.

In [ ]:
def engineer_features(df):
    feat_df = pd.DataFrame()
    feat_df['T'] = df['temperature_c']
    feat_df['RH'] = df['humidity_pct']
    feat_df['P'] = df['pressure_hpa']
    feat_df['Td'] = df['dew_point_c']
    feat_df['dT_dt'] = df['temperature_c'].diff().fillna(0.0)
    feat_df['dP_dt'] = df['pressure_hpa'].diff().fillna(0.0)
    feat_df['dRH_dt'] = df['humidity_pct'].diff().fillna(0.0)
    feat_df['drop_flag'] = 0.0
    return feat_df

train_feat = engineer_features(df_train)
feature_names = list(train_feat.columns)

means = train_feat.mean().to_dict()
stds = train_feat.std().replace(0, 1.0).to_dict()

scaler_config = {
    "feature_names": feature_names,
    "means": {k: float(v) for k, v in means.items()},
    "stds": {k: float(v) for k, v in stds.items()},
    "threshold": 0.042
}

with open("scaler.json", "w") as f:
    json.dump(scaler_config, f, indent=2)
print("Exported scaler.json with baseline statistics.")

norm_train = (train_feat - pd.Series(means)) / pd.Series(stds)
display(norm_train.describe().round(3))

### Step 4: Create Rolling Time Windows (Window Size = 12)

In [ ]:
def create_windows(data_arr, window_size=12):
    windows = []
    for i in range(len(data_arr) - window_size + 1):
        windows.append(data_arr[i:i + window_size])
    return np.array(windows, dtype=np.float32)

X_train_raw = create_windows(norm_train.values, window_size=12)
# PyTorch Conv1D expects shape: (batch_size, num_channels, sequence_length)
X_train = np.transpose(X_train_raw, (0, 2, 1))
print(f"Training Windows Shape: {X_train.shape} -> (batches, {X_train.shape[1]} channels, {X_train.shape[2]} timesteps)")

train_dataset = TensorDataset(torch.tensor(X_train))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

### Step 5: Define Lightweight 1D-CNN Autoencoder Architecture

In [ ]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, in_channels=8, seq_len=12, latent_dim=8):
        super().__init__()
        # Encoder
        self.enc_conv1 = nn.Conv1d(in_channels, 16, kernel_size=3, padding=1)
        self.enc_relu1 = nn.ReLU()
        self.enc_pool = nn.MaxPool1d(2) # 12 -> 6
        self.enc_conv2 = nn.Conv1d(16, latent_dim, kernel_size=3, padding=1)
        self.enc_relu2 = nn.ReLU()
        
        # Decoder
        self.dec_upsample = nn.Upsample(scale_factor=2, mode='nearest') # 6 -> 12
        self.dec_conv1 = nn.Conv1d(latent_dim, 16, kernel_size=3, padding=1)
        self.dec_relu1 = nn.ReLU()
        self.dec_conv2 = nn.Conv1d(16, in_channels, kernel_size=3, padding=1)

    def forward(self, x):
        # x shape: (B, 8, 12)
        z = self.enc_relu1(self.enc_conv1(x))
        z = self.enc_pool(z)
        z = self.enc_relu2(self.enc_conv2(z))
        
        out = self.dec_upsample(z)
        out = self.dec_relu1(self.dec_conv1(out))
        out = self.dec_conv2(out)
        return out

model = Conv1DAutoencoder().to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {num_params:,} (~{num_params * 4 / 1024:.1f} KB in float32)")

### Step 6: Train Model and Track Loss Curve

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-5)
epochs = 20
loss_history = []

model.train()
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    for batch in train_loader:
        x_batch = batch[0].to(device)
        optimizer.zero_grad()
        reconstruction = model(x_batch)
        loss = criterion(reconstruction, x_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(x_batch)
        
    epoch_loss /= len(train_dataset)
    loss_history.append(epoch_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs}] Reconstruction Loss (MSE): {epoch_loss:.6f}")

### Step 7: Visual Diagnostic Charts
- Chart 1: Training Loss Convergence Curve
- Chart 2: Normal Weather MSE vs Fault Injection Spikes
- Chart 3: Atmospheric Multivariate Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Training Loss Curve
axes[0].plot(range(1, epochs + 1), loss_history, marker='o', color='#06b6d4', linewidth=2)
axes[0].set_title("Chart 1: Training Loss Curve (MSE Convergence)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Reconstruction Loss")
axes[0].grid(True, linestyle='--', alpha=0.6)

# Chart 2: Normal Error vs Anomaly Detection Gap
model.eval()
with torch.no_grad():
    sample_x = torch.tensor(X_train[:300]).to(device)
    normal_pred = model(sample_x)
    normal_mse = torch.mean((sample_x - normal_pred) ** 2, dim=(1, 2)).cpu().numpy()

drift_x = sample_x.clone()
drift_x[:, 1, :] += 1.8 # Injected +15% humidity drift
with torch.no_grad():
    drift_pred = model(drift_x)
    drift_mse = torch.mean((drift_x - drift_pred) ** 2, dim=(1, 2)).cpu().numpy()

axes[1].hist(normal_mse, bins=25, alpha=0.7, color='#10b981', label='Nominal Weather (< 0.02)')
axes[1].hist(drift_mse, bins=25, alpha=0.7, color='#f43f5e', label='Capacitive Drift (> 0.042)')
axes[1].axvline(0.042, color='red', linestyle='--', linewidth=2, label='Threshold = 0.042')
axes[1].set_title("Chart 2: Anomaly Detection Gap (Clear Margin)", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Reconstruction Error (MSE)")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

# Chart 3: Correlation Heatmap
corr_matrix = train_feat[['T', 'RH', 'P', 'Td', 'dT_dt', 'dP_dt']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[2], fmt='.2f')
axes[2].set_title("Chart 3: Multivariate Physical Correlation", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### Step 8: Export Model to ONNX (< 500 KB)
Exports `autoencoder.onnx` so it can run at high speed in FastAPI or on edge devices.

In [ ]:
model.eval()
dummy_input = torch.randn(1, 8, 12, requires_grad=False).to(device)
onnx_path = "autoencoder.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['weather_window'],
    output_names=['reconstructed_window'],
    dynamic_axes={'weather_window': {0: 'batch_size'}, 'reconstructed_window': {0: 'batch_size'}}
)

size_kb = os.path.getsize(onnx_path) / 1024.0
print(f"SUCCESS: Exported {onnx_path}!")
print(f"File Size: {size_kb:.1f} KB (Well below 500 KB limit!)")
print("\nDownload both 'autoencoder.onnx' and 'scaler.json' and place into: backend/ml_artifacts/")